# Part 3 — BERT-Style Robust NLP Classification (Colab GPU)

## Setup
1. **Runtime → Change runtime type → GPU** (T4 is sufficient)
2. Upload `part3_BERT_Robust_NLP_Experiments.py` to Colab files panel (left sidebar)
3. Run all cells top-to-bottom

## What runs
| Dataset | Model | Classes |
|---------|-------|---------|
| MedMCQA | BiomedBERT | 4 |
| MedQA-USMLE | BioBERT | 4 |
| PubMedQA | SciBERT | 3 |

8 loss functions × 4 batteries (clean / uniform noise / class-dep noise / GCE sweep)

In [ ]:
# ── Cell 1: Install pinned dependencies ───────────────────────────────────────
!pip install -q --upgrade \
    typing_extensions>=4.10.0 \
    torch>=2.6.0 \
    torchvision>=0.21.0 \
    transformers>=4.46.0 \
    datasets>=2.20.0 \
    accelerate>=0.33.0 \
    huggingface_hub>=0.24.0 \
    scikit-learn>=1.3.0 \
    seaborn>=0.13.0 \
    matplotlib>=3.8.0 \
    pandas>=2.1.0 \
    tqdm>=4.66.0
print('✓ Dependencies installed (pinned).')
print('If imports fail in same session: Runtime -> Restart session once.')


In [ ]:
# ── Cell 2: Runtime sanity + GPU check ───────────────────────────────────────
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return 'not-installed'

print('typing_extensions:', ver('typing_extensions'))
print('torch:', ver('torch'))
print('transformers:', ver('transformers'))
print('datasets:', ver('datasets'))

import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'✓ GPU : {dev.name}  VRAM = {dev.total_memory / 1024**3:.1f} GB')
    print(f'  CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')
else:
    print('⚠ No GPU — go to Runtime → Change runtime type → GPU')


In [ ]:
# ── Cell 3: Upload script ────────────────────────────────────────────────────
import os
SCRIPT = 'part3_BERT_Robust_NLP_Experiments.py'

if not os.path.exists(SCRIPT):
    from google.colab import files
    print(f'Please upload {SCRIPT}')
    uploaded = files.upload()
    assert SCRIPT in uploaded, f'Expected {SCRIPT}, got {list(uploaded.keys())}'

print(f'✓ {SCRIPT} ready ({os.path.getsize(SCRIPT)/1024:.0f} KB)')

In [ ]:
# ── Cell 4: (Optional) Set HF token for gated models ─────────────────────────
# Uncomment ONLY if needed:
# from google.colab import userdata
# import os
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# ── Cell 5: RUN the full experiment pipeline ─────────────────────────────────
# Batteries A–D: clean → uniform noise → class-dep noise → GCE(q) sweep
# QUICK_RUN=True (default) takes ~30-60 min on T4.
import runpy
runpy.run_path(SCRIPT, run_name='__main__')

In [ ]:
# ── Cell 6: List results ─────────────────────────────────────────────────────
from pathlib import Path
rd = Path('results_bert')
if rd.exists():
    for f in sorted(rd.iterdir()):
        print(f'  {f.name:45s}  {f.stat().st_size/1024:7.1f} KB')
else:
    print('Results folder not yet created — run Cell 5 first.')

In [ ]:
# ── Cell 7: Preview summary table ────────────────────────────────────────────
import pandas as pd
sp = Path('results_bert') / 'summary_all.csv'
if sp.exists():
    df = pd.read_csv(sp)
    print(df[['dataset','loss','noise_type','noise_rate','acc_str']].to_string(index=False))
else:
    print('Not yet generated — run Cell 5 first.')

In [ ]:
# ── Cell 8: Download results ─────────────────────────────────────────────────
import shutil
from google.colab import files
rd = Path('results_bert')
if rd.exists():
    arc = shutil.make_archive('results_bert', 'zip', '.', 'results_bert')
    files.download(arc)
else:
    print('No results to download yet.')